<a href="https://colab.research.google.com/github/Kaviyarasi-Sasiperumal/AI_-Based_-Document-_Search_-and-_Knowledge-_Retrieval_-with-_Conversational_Interface/blob/main/milestone_4_deployment____final_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pypdf sentence-transformers faiss-cpu transformers gradio torch


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.2/331.2 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 68.1 MB/s eta 0:00:00


In [ ]:
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from transformers import pipeline
import faiss
import numpy as np
import gradio as gr
import time


In [ ]:
documents = []
metadata = []
embedder = SentenceTransformer("all-MiniLM-L6-v2")
index = None
llm = pipeline(
    "text-generation",
    model="google/flan-t5-base",
    max_new_tokens=200
)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLl

In [ ]:
def chunk_text(text, chunk_size=250, overlap=100):
    chunks = []
    for i in range(0, len(text), chunk_size - overlap):
        chunks.append(text[i:i+chunk_size])
    return chunks


def load_document(file):
    global documents, metadata, index

    documents = []
    metadata = []

    reader = PdfReader(file.name)

    for page_num, page in enumerate(reader.pages):
        text = page.extract_text()
        if text:
            chunks = chunk_text(text)
            for chunk in chunks:
                documents.append(chunk)
                metadata.append(f"{file.name} - page {page_num+1}")

    if not documents:
        return "❌ No readable text found in document."


    embeddings = embedder.encode(documents)
    index = faiss.IndexFlatL2(embeddings.shape[1])
    index.add(np.array(embeddings))

    return f"✅ Document loaded successfully! Pages indexed: {len(metadata)}"


In [ ]:
!pip install gradio PyPDF2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 5.5 MB/s eta 0:00:00


In [ ]:
import gradio as gr
import PyPDF2
import time

document_text = ""
chunks = []
chat_history_data = []

# -------- CHUNK FUNCTION --------
def create_chunks(text, chunk_size=300):
    return [text[i:i+chunk_size] for i in range(0, len(text), chunk_size)]


# -------- LOAD MULTIPLE DOCUMENTS --------
def load_document(files):
    global document_text, chunks
    document_text = ""
    chunks = []

    if not files:
        return "❌ No documents uploaded", ""

    total_pages = 0

    for file in files:
        text = ""

        if file.name.lower().endswith(".pdf"):
            reader = PyPDF2.PdfReader(file)
            total_pages += len(reader.pages)
            for page in reader.pages:
                t = page.extract_text()
                if t:
                    text += t + "\n"

        elif file.name.lower().endswith(".txt"):
            text += file.read().decode("utf-8")
            total_pages += 1

        document_text += text + "\n"

    chunks = create_chunks(document_text)

    stats = f"""
📊 DOCUMENT STATISTICS

📄 Total Documents : {len(files)}
📄 Total Pages     : {total_pages}
🧩 Total Chunks    : {len(chunks)}
📝 Total Words     : {len(document_text.split())}
🔡 Total Characters: {len(document_text)}
"""
    return "✅ Documents processed successfully", stats


# -------- ANSWER EXTRACTION --------
def get_answer_from_document(question):
    lines = [l.strip() for l in document_text.split("\n") if l.strip()]
    q = question.lower().strip()

    field_keys = ["degree", "branch", "specialization", "name", "email", "phone", "qualification"]
    for key in field_keys:
        if key in q:
            for line in lines:
                if line.lower().startswith(key) and ":" in line:
                    return line.split(":", 1)[1].strip()

    if "skill" in q:
        skills = [l.strip("•- ").strip()
                  for l in lines
                  if l.startswith(("•", "-", "•"))]
        if skills:
            return ", ".join(skills)

    if "objective" in q:
        for i, line in enumerate(lines):
            if line.lower() == "objective":
                answer = ""
                j = i + 1
                while j < len(lines):
                    answer += lines[j] + " "
                    if lines[j].endswith("."):
                        break
                    j += 1
                return answer.strip()

    return "Sorry, the exact answer was not found in the document."


# -------- CHAT FUNCTION --------
def handle_message(user_input, history):
    global chat_history_data

    if history is None:
        history = []

    if not user_input:
        return history, ""

    start = time.perf_counter()
    msg = user_input.lower().strip()

    if msg in ["hi", "hello", "hey"]:
        reply = "Hello 👋 How can I help you?"
    elif not document_text:
        reply = "📄 Please upload documents first."
    else:
        answer = get_answer_from_document(user_input)
        response_time = round((time.perf_counter() - start) * 1000, 2)
        reply = f"{answer}\n\n⏱ Response Time: {response_time} ms"

    # IMPORTANT FIX (no mutation bug)
    history = history + [
        {"role": "user", "content": user_input},
        {"role": "assistant", "content": reply},
    ]

    chat_history_data.append(f"User: {user_input}\nBot: {reply}\n\n")

    return history, ""


# -------- CLEAR CHAT FUNCTION --------
def clear_chat():
    global chat_history_data
    chat_history_data = []
    return []


# -------- DOWNLOAD CHAT FUNCTION --------
def download_chat():
    if not chat_history_data:
        return None

    filename = "chat_history.txt"
    with open(filename, "w", encoding="utf-8") as f:
        f.writelines(chat_history_data)

    return filename


# -----------------------
# UI
# -----------------------
with gr.Blocks() as demo:

    gr.Markdown("## 🤖 Multi-Document Chatbot")

    with gr.Row():

        # LEFT PANEL
        with gr.Column(scale=1):

            gr.Markdown("### 📂 Upload PDF / TXT")
            file_input = gr.File(file_count="multiple")

            status = gr.Textbox(label="Status", interactive=False)
            stats_box = gr.Textbox(label="Document Statistics", lines=8, interactive=False)

            file_input.change(load_document, file_input, [status, stats_box])

            clear_btn = gr.Button("🗑 Clear Chat")
            download_btn = gr.Button("⬇ Download Chat")
            download_file = gr.File()

        # RIGHT PANEL
        with gr.Column(scale=3):

            chatbot = gr.Chatbot(
                height=450,
                type="messages"
            )

            user_input = gr.Textbox(placeholder="Type your message...")
            send_btn = gr.Button("Send")

            send_btn.click(
                handle_message,
                inputs=[user_input, chatbot],
                outputs=[chatbot, user_input]
            )

            user_input.submit(
                handle_message,
                inputs=[user_input, chatbot],
                outputs=[chatbot, user_input]
            )

    clear_btn.click(clear_chat, outputs=chatbot)
    download_btn.click(download_chat, outputs=download_file)

demo.launch()

/tmp/ipython-input-2315311771.py:169: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://17814005f82b893c1a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
